In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

### **DATA READING**

In [0]:
df = spark.read.format("parquet")\
            .load("abfss://bronze@customersprojectete.dfs.core.windows.net/products")

In [0]:
df.limit(5).display()

In [0]:
df = df.drop("_rescued_data")

In [0]:
df.limit(5).display()

In [0]:
df.createOrReplaceTempView("products")

### **FUNCTIONS**

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cata.bronze.C(product_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN product_price * 0.90

In [0]:
%sql
SELECT product_id, price, databricks_cata.bronze.discounted_price(price) FROM products

In [0]:
df = df.withColumn("discounted_price", expr("databricks_cata.bronze.discounted_price(price)"))
df.display()

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cata.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
return p_brand.upper()
$$;

In [0]:
%sql
select product_id, databricks_cata.bronze.upper_func(brand) from products

In [0]:
df.withColumn("upper_brand", expr("databricks_cata.bronze.upper_func(brand)")).display()

In [0]:
df.write.mode("overwrite")\
    .format("delta")\
    .save("abfss://silver@customersprojectete.dfs.core.windows.net/products")

In [0]:
%sql
create table if not exists databricks_cata.silver.products_silver
using delta
location "abfss://silver@customersprojectete.dfs.core.windows.net/products"

In [0]:
%sql
select * from databricks_cata.silver.products_silver